# Cascade tests · which profile measurable matches the fit parameter K?
**Question.** The fit is δ_a = K ln(RH0/RH) with K = Γ_h·C. We want a number read off the profiles that gives K, so the story lines up. The sickle slope (A) alone is about twice too big, so A = B can be dropped if something else works.

**Sign.** K = −dy/d ln RH over RH 0.30–0.60, so K < 0 when δ_a grows with RH. C = −dz_LCL/d ln RH > 0.

**Needs** `qsub side_tests/run_cascade_composites.pbs`. Fixed pressure levels, terrain below 0.5 km, unweighted. Kernel `npl`, run from `side_tests/`. Each section ends with what to look for; none of them has been read on real data yet.

In [ ]:
%matplotlib inline
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
HERE = pathlib.Path.cwd()
REPO = HERE.parent
sys.path[:0] = [str(HERE), str(REPO / "src")]
import numpy as np, matplotlib.pyplot as plt
import pandas as pd
from tpl_mll.config import load
import cascade_tests as T
pd.set_option("display.width", 200, "display.max_columns", 40, "display.float_format", "{:.2f}".format)

YEARS = None                      # e.g. (1980, 1999) to split the record
A = T.load_all(pathlib.Path(load("paths")["work"]) / "side_tests" / "casc", years=YEARS)
print({e: a["n_files"] for e, a in A.items()}, "files per event")
names = list(A["T"]["names"])
CASC = {e: [n for n in A[e]["names"] if n.startswith(e + "_")] for e in A}
COL = {"T_MLL": "#e57373", "T_TPL": "#b71c1c", "M_MLL": "#64b5f6", "M_TPL": "#0d47a1", "all": "k"}

In [ ]:
# every group: the four boxes, "all" (T, M, T+M), and the grid boxes for T, M and T+M
GROUPS = {}
for e, a in A.items():
    for n in CASC[e]:
        GROUPS[n] = T.group(a, n)
    GROUPS[f"all_{e}"] = T.group(a, "all")
GROUPS["all"] = T.group(A["T"], "all", A["M"])
GRID = {}
grid_names = [n for n in names if n.startswith("g")]
for n in grid_names:
    for key, c in (("T", T.group(A["T"], n)), ("M", T.group(A["M"], n)), ("T+M", T.group(A["T"], n, A["M"]))):
        if ((c["n"] >= T.MIN_N) & (c["RH_2m"] >= T.WINDOW[0]) & (c["RH_2m"] <= T.WINDOW[1])).sum() >= T.MIN_BINS:
            GRID[(key, n)] = c
MEAS = {k: T.measures(c) for k, c in GROUPS.items()}
RES = {k: T.candidates(GROUPS[k], MEAS[k]) for k in GROUPS}
GRES = [dict(T.candidates(c), event=k[0], box=k[1]) for k, c in GRID.items()]
print(len(GROUPS), "named groups,", len(GRES), "grid boxes usable")

## 1 · The target: K, C and the required slope
`Γ_req = K_hLPL / C` is the locus slope B. Also reconciles Task 5 with the old notes: `Γ_z` regresses h_LPL on the actual z_LCL (Task 5), `plateau` is −(RH/2 km)·dδ_a/dRH (the notes).

**Look for:** K_hstar small next to K_hLPL in the boxes, large in `all` if the T/M mix matters. C near 2 km. Whether Γ_z and the plateau agree; if not, that explains why Task 5 gave −1.66 for M_MLL against −4.3 or steeper in the notes. Pooled log K should be close to −10.5 (Γ_h = K/2 km = −5.3 in the figure facts; `pooled_hstar.py` gave −10.53). This mask is low terrain only, so expect some difference.

In [ ]:
rows = []
for k, c in GROUPS.items():
    r = RES[k]
    ok = (c["n"] >= T.MIN_N) & (c["RH_2m"] >= T.WINDOW[0]) & (c["RH_2m"] <= T.WINDOW[1])
    gz = np.polyfit(c["zLCL"][ok], c["h_LPL"][ok], 1, w=np.sqrt(c["n"][ok]))[0] if ok.sum() >= T.MIN_BINS else np.nan
    rh, d = c["RH_2m"], c["delta_a"]
    local = np.gradient(d[ok], rh[ok]) * (-rh[ok] / 2.0) if ok.sum() >= T.MIN_BINS else np.array([np.nan])
    rows.append(dict(group=k, n=r["n"], K=r["K"], K_hLPL=r["K_hLPL"], K_hstar=r["K_hstar"], C=r["C"], C_kink=r["C_kink"],
                     Gamma_req=r["Gamma_req"], Gamma_z=gz, plateau=np.nanmedian(local)))
pd.DataFrame(rows).set_index("group")

## 2 · Are the fits sensible on the profiles?
Composites offset by RH bin, with the three-segment fit (thin black), kink (o), break (▲) and LCL (×).

**Look for:** kinks on the visible corner, the LCL near the kink (it often isn't), and fits that have not relabelled a segment.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(17, 6), sharey=True)
for ax, k in zip(axs, [*CASC.get("T", []), *CASC.get("M", [])]):
    c, m = GROUPS[k], MEAS[k]
    off = 0
    for i in np.nonzero(c["n"] >= 500)[0]:
        h, z = c["MSE"][i], c["z"][i]
        keep = (c["levels"] >= 350) & np.isfinite(h)
        o = np.argsort(z[keep])
        ax.plot(h[keep][o] + off, z[keep][o], color=plt.cm.viridis(c["RH_2m"][i]), lw=1.5)
        if np.isfinite(m["G_h"][i]):
            zk = m["z_kink"][i]
            ax.plot(m["h_kink"][i] + off, zk, "ko", ms=4)
            zb = T.interp_p(c["levels"].astype(float), z, m["p_break"][i])
            ax.plot(m["h_kink"][i] + m["G_h"][i] * (zb - zk) + off, zb, "k^", ms=4)
            ax.plot([m["h_kink"][i] + off, m["h_kink"][i] + m["G_h"][i] * (zb - zk) + off], [zk, zb], "k-", lw=0.7)
            ax.plot([m["h_kink"][i] + off, m["h_kink"][i] - m["G_bl"][i] * zk + off], [zk, 0], "k-", lw=0.7)
        ax.plot(m["h_at_LCL"][i] + off, c["zLCL"][i], "rx", ms=6)
        off += 8
    ax.set(title=k, xlabel="MSE + offset [kJ/kg]", ylim=(0, 8))
axs[0].set_ylabel("z above surface [km]")

## 3 · A against B, per RH and per box
Top row: ML slope and sickle slope against RH, the old fits drawn in. Bottom: kink depth against the LCL, with the coupling.

**Hypothesis 1:** B/A ≈ coupling, because the kink moves less than the LCL.
**Hypothesis 2:** B ≈ A − Γ_BL, because h_LPL = h_kink − Γ_BL·D, so a tilted layer weakens the locus slope. Earlier work found adding Γ_BL made things worse, so this is a recheck with the sign written out.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 8))
for k in [*CASC.get("T", []), *CASC.get("M", [])]:
    c, m = GROUPS[k], MEAS[k]
    axs[0, 0].plot(c["RH_2m"], m["G_bl"], "-o", ms=3, color=COL[k], label=k)
    axs[0, 1].plot(c["RH_2m"], m["G_h"], "-o", ms=3, color=COL[k])
    axs[1, 0].plot(c["zLCL"], m["z_kink"], "-o", ms=3, color=COL[k])
    axs[1, 1].plot(c["RH_2m"], m["p_kink"], "-o", ms=3, color=COL[k])
    axs[1, 1].plot(c["RH_2m"], m["p_break"], ":", color=COL[k])
x = np.linspace(.1, .9, 5); axs[0, 0].plot(x, -9.03 * x + 1.63, "k--", label="−9.03 RH + 1.63 (notes)")
axs[0, 0].set(xlabel="RH", ylabel="Γ_BL [kJ/kg/km]"); axs[0, 0].legend(fontsize=7)
axs[0, 1].set(xlabel="RH", ylabel="Γ_h sickle [kJ/kg/km]")
axs[1, 0].plot([0, 5], [0, 5], "k:"); axs[1, 0].set(xlabel="z_LCL [km]", ylabel="kink depth D [km]")
axs[1, 1].set(xlabel="RH", ylabel="kink (—), break (··) [hPa]"); axs[1, 1].invert_yaxis()
plt.tight_layout()
rows = []
for k in [*CASC.get("T", []), *CASC.get("M", []), "all_T", "all_M", "all"]:
    r = RES[k]
    rows.append(dict(group=k, A=r["G_h"], BL=r["G_bl"], B=r["Gamma_req"], B_over_A=r["Gamma_req"] / r["G_h"], coupling=r["coupling"],
                     tilt_over_A=(r["G_h"] - r["G_bl"]) / r["G_h"]))
pd.DataFrame(rows).set_index("group")

## 4 · Exact budgets of K_hLPL
Each row adds up by construction, so this shows where B comes from.
- at LCL + gap: energy of the profile at the LCL, plus the tilt between the LPL and the LCL
- at kink + gap: the same, using the fitted kink
- slide + shift: `K_slide` keeps one moist profile fixed and moves only the LCL (the pure picture). `K_shift` is the whole profile changing with RH at fixed pressure. `K_at_LCL = slide + shift`.

On the known-answer test the shift resolves to about ±0.5, so smaller values mean nothing.

**Look for:** if `K_slide` ≈ 2 × K_hLPL, the fixed-profile picture really does overshoot, and the gap and shift terms are what cancel half of it. Those terms are then the physics to explain.

In [ ]:
cols = ["K_hLPL", "K_at_LCL", "K_gap_LCL", "K_at_kink", "K_gap_kink", "K_slide", "K_shift"]
pd.DataFrame({k: {x: RES[k][x] for x in cols} for k in [*CASC.get("T", []), *CASC.get("M", []), "all_T", "all_M", "all"]}).T

## 5 · How does the profile move with RH at fixed pressure?
K(p) = −dy/d ln RH at each level, split into h, c_pT and L_v q (the gz part is what is left). Negative means the value rises with RH.

**Look for:** whether the lower free troposphere moistens with RH (`K_shift` < 0) or dries (> 0), and which layer. The notes say moisture dominates both slopes.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 5), sharey=True, sharex=True)
for ax, k in zip(axs, [*CASC.get("T", []), *CASC.get("M", [])]):
    c = GROUPS[k]
    lev = c["levels"].astype(float)
    K = {"h": [], "cpT": [], "Lq": []}
    for j in range(len(lev)):
        h, t, z = c["MSE"][:, j], c["T"][:, j], c["Z"][:, j]
        K["h"].append(T.kfit(c, h)); K["cpT"].append(T.kfit(c, T.CP * t)); K["Lq"].append(T.kfit(c, h - T.CP * t - z))
    for v, col in (("h", "k"), ("cpT", "#e53935"), ("Lq", "#1e88e5")):
        ax.plot(K[v], lev, color=col, label=v)
    ax.axvline(0, color="0.7"); ax.axhline(np.nanmean(c["p_LCL_LPL"]), color="0.7", ls=":")
    ax.set(title=k, xlabel="K(p) [kJ/kg]", ylim=(1000, 300))
axs[0].set_ylabel("p [hPa]"); axs[0].legend()

## 6 · Does entrainment give K? (the link to Section 5)
`ε_req` is the ε that brings a plume through the composite to h*500 at 500 hPa. If it is constant across RH, then δ_a ≈ ε × area, and K can be read from the profile geometry at one ε, which would tie Section 5 to the RH fit. Composites smooth the events, so this is not the same plume as stage f.

**Look for:** a flat ε_req near 0.2 km⁻¹, and `dil` (dilution at 500 hPa at ε = 0.2) following δ_a, turnover included.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for k in [*CASC.get("T", []), *CASC.get("M", []), "all"]:
    c, m = GROUPS[k], MEAS[k]
    axs[0].plot(c["RH_2m"], m["eps_req"], "-o", ms=3, color=COL[k], label=k)
    axs[1].plot(c["RH_2m"], c["delta_a"], "-", color=COL[k], label=k)
    axs[1].plot(c["RH_2m"], m["dil_500"], "--", color=COL[k])
axs[0].axhline(T.EPS, color="0.6", ls=":"); axs[0].set(xlabel="RH", ylabel="ε_req [km⁻¹]", ylim=(0, 1)); axs[0].legend(fontsize=7)
axs[1].set(xlabel="RH", ylabel="δ_a (—), dilution at 500 hPa (--) [kJ/kg]")
pd.DataFrame({k: dict(K=RES[k]["K"], K_dil=RES[k]["K_dil"], eps_req=RES[k]["eps_req"]) for k in [*CASC.get("T", []), *CASC.get("M", []), "all"]}).T

## 7 · Scoreboard: which candidate tracks K across every grid box?
Each grid box is 3 kJ/kg wide in h*500 and h500, as the cascade boxes are. A good measurable has ratio near 1, high R² and low RMSE, for T and M alike. `K_hLPL` is the check that K really is h_LPL inside boxes; it isn't a candidate, since it's nearly K itself.

**Look for:** a candidate that needs no h_LPL and beats a plain constant. The `constant` row is how well the mean K alone does, so anything with larger RMSE adds nothing.

In [ ]:
rows = []
for ev in ("T", "M", "T+M"):
    sub = [r for r in GRES if r["event"] == ev]
    Ks = np.array([r["K"] for r in sub], float)
    Ks = Ks[np.isfinite(Ks)]
    if len(Ks):
        rows.append(dict(event=ev, candidate="constant", n=len(Ks), r2=0.0, rmse=Ks.std(), ratio=1.0))
    for key, lab in T.CANDS.items():
        rows.append(dict(event=ev, candidate=lab, **T.score(sub, key)))
board = pd.DataFrame(rows)
display(board.pivot(index="candidate", columns="event", values=["ratio", "r2", "rmse"]).sort_values(("rmse", "T+M")))
keys = list(T.CANDS)
fig, axs = plt.subplots(2, (len(keys) + 1) // 2, figsize=(3.3 * ((len(keys) + 1) // 2), 6.5), sharex=True, sharey=True)
for ax, key in zip(axs.flat, keys):
    for ev, col in (("T", "#e53935"), ("M", "#1e88e5")):
        s = [r for r in GRES if r["event"] == ev]
        ax.plot([r["K"] for r in s], [r[key] for r in s], ".", color=col, ms=5)
    lim = (-35, 5); ax.plot(lim, lim, "k:"); ax.set(xlim=lim, ylim=lim, title=T.CANDS[key], xlabel="K")
    ax.title.set_fontsize(8)
plt.tight_layout()

## 8 · Is one K enough? Local slope against RH
Local dδ_a/dz_LCL across RH, for the event types and boxes. A constant slope gives the log form. A slope proportional to RH gives a straight line in RH, since dδ_a/dRH = (−aRH)(−C/RH) = aC. The pooled tables suggest T follows −13 RH (dashed) while M sits near a constant −4.2 (dotted) over RH 0.35–0.6, with both dropping to zero at the turnover. The boxes hold h*500 fixed, so they show whether this survives without the ceiling term.

Right panel: the same for h_LPL alone, next to the sickle minus ML slope.

**Look for:** whether the boxes split the same way, T ∝ RH and M constant. If they do, ask what the RH-proportional part is; the ML slope Γ_BL ≈ −9 RH + 1.6 has the same form.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
STY = {"all_T": ("#e53935", "--"), "all_M": ("#1e88e5", "--"), "all": ("k", "-")}
for k in [*CASC.get("T", []), *CASC.get("M", []), "all_T", "all_M", "all"]:
    c, m = GROUPS[k], MEAS[k]
    col, ls = STY.get(k, (COL.get(k, "0.5"), "-"))
    ok = (c["n"] >= T.MIN_N) & np.isfinite(c["zLCL"])
    if ok.sum() >= 3:
        axs[0].plot(c["RH_2m"][ok], np.gradient(c["delta_a"][ok], c["zLCL"][ok]), ls, marker="o", ms=3, color=col, label=k)
        axs[1].plot(c["RH_2m"][ok], np.gradient(c["h_LPL"][ok], c["zLCL"][ok]), ls, marker="o", ms=3, color=col)
        axs[1].plot(c["RH_2m"][ok], (m["G_h"] - m["G_bl"])[ok], ":", color=col)
x = np.linspace(.05, .7, 5)
axs[0].plot(x, -13 * x, "k--", lw=1, label="−13 RH"); axs[0].axhline(-4.2, color="k", ls=":", lw=1, label="−4.2")
for ax, t in zip(axs, ("local dδ_a/dz_LCL", "local dh_LPL/dz_LCL (—), sickle − ML slope (··)")):
    ax.axvspan(*T.WINDOW, color="#fff59d", alpha=.3); ax.axhline(0, color="0.6")
    ax.set(xlabel="RH", title=t, ylim=(-20, 10), xlim=(0, 1))
axs[0].set_ylabel("kJ/kg/km"); axs[0].legend(fontsize=7)

## 9 · Put a winner to work
Set `BEST` from section 7, then draw the δ_a it predicts with no fitting: K comes only from the profile measurable, RH0 from the data. This is the plot the story needs to hold.

In [ ]:
BEST = "K_tilt_kink"                  # any key of T.CANDS
fig, axs = plt.subplots(1, 5, figsize=(18, 3.6), sharey=True)
for ax, k in zip(axs, [*CASC.get("T", []), *CASC.get("M", []), "all"]):
    c, r = GROUPS[k], RES[k]
    ok = c["n"] >= T.MIN_N
    x, d = c["RH_2m"][ok], c["delta_a"][ok]
    w = (x >= T.WINDOW[0]) & (x <= T.WINDOW[1])
    ax.plot(x, d, "o", ms=4, color=COL[k])
    if w.sum() >= T.MIN_BINS and np.isfinite(r[BEST]):
        anchor = d[w].mean() + r[BEST] * np.log(x[w]).mean()          # passes through the window mean
        xs = np.linspace(.05, .95, 100)
        ax.plot(xs, anchor - r[BEST] * np.log(xs), "k-", label=f"{BEST} = {r[BEST]:.1f}")
        ax.plot(xs, anchor - r["K"] * np.log(xs), "k:", label=f"fit K = {r['K']:.1f}")
    ax.axvspan(*T.WINDOW, color="#fff59d", alpha=.3); ax.set(title=k, xlabel="RH", xlim=(0, 1)); ax.legend(fontsize=7)
axs[0].set_ylabel("δ_a [kJ/kg]")

## 10 · The closed form: δ_a from the deficit against the weighted environment
The plume is linear, so h_c(500) = (1 − f) h_LPL + f h̄_e with f = 1 − exp(−εL), where h̄_e is the environment weighted toward the top. Setting h_c(500) = h*500 gives

**δ_a = (exp(εL) − 1)(h*500 − h̄_e)**

h_LPL isn't an explicit input, so unlike section 6 this is a real test. The mixed layer still enters through h̄_e. The formula reduces to Duan- and Jha-style subsaturation bounds for small εL. Composites include stable events, so expect trouble near RH0.

**Look for:** (a) predicted curves on δ_a, for one ε across boxes. (b) K_closed against K, with a ratio near 1 and RMSE below the  row. (c) Does the deficit grow from the part above the LCL (the geometric story)? The part below the LCL is usually negative, since the mixed layer sits above h*500, and a warmer, moister layer makes it more negative, which pushes the other way.

In [ ]:
CL = {k: T.closed(GROUPS[k]) for k in GROUPS}
CRES = {k: T.closed_candidates(GROUPS[k], CL[k]) for k in GROUPS}
GCL = [dict(T.closed_candidates(c), event=k[0], box=k[1]) for k, c in GRID.items()]
named = [*CASC.get("T", []), *CASC.get("M", []), "all_T", "all_M", "all"]
pd.DataFrame({k: {"K": CRES[k]["K"], **{f"K({e:g})": CRES[k][f"K_closed_{e:g}"] for e in T.EPS_LIST}} for k in named}).T

In [ ]:
# (a) predicted against actual δ_a, RH by RH
EPS_SHOW = (0.10, 0.20, 0.30)
fig, axs = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
for ax, k in zip(axs.flat, named):
    c = GROUPS[k]
    ok = c["n"] >= T.MIN_N
    ax.plot(c["RH_2m"][ok], c["delta_a"][ok], "ko-", ms=3, lw=2, label="δ_a")
    for e, col in zip(EPS_SHOW, ("#90caf9", "#1e88e5", "#0d47a1")):
        ax.plot(c["RH_2m"][ok], CL[k]["pred"][T.EPS_LIST.index(e)][ok], color=col, label=f"ε = {e:g}")
    ax.axvspan(*T.WINDOW, color="#fff59d", alpha=.3); ax.axhline(0, color="0.7"); ax.set(title=k, ylim=(-20, 25), xlim=(0, 1))
axs[0, 0].legend(fontsize=7); axs[1, 0].set(xlabel="RH", ylabel="kJ/kg")
for ax in axs[1]: ax.set_xlabel("RH")
plt.tight_layout()

In [ ]:
# (b) scoreboard across the grid boxes, for K and for the δ_a level itself
rows = []
for ev in ("T", "M", "T+M"):
    sub = [r for r in GCL if r["event"] == ev]
    Ks = np.array([r["K"] for r in sub], float); Ks = Ks[np.isfinite(Ks)]
    rows.append(dict(event=ev, eps="constant", n=len(Ks), r2=0.0, rmse=Ks.std() if len(Ks) else np.nan, ratio=1.0, level_rmse=np.nan))
    for e in T.EPS_LIST:
        lv = np.array([r[f"rmse_{e:g}"] for r in sub], float)
        rows.append(dict(event=ev, eps=f"{e:g}", **T.score(sub, f"K_closed_{e:g}"), level_rmse=np.sqrt(np.nanmean(lv ** 2)) if np.isfinite(lv).any() else np.nan))
display(pd.DataFrame(rows).set_index(["event", "eps"]))
show = (0.10, 0.20, 0.30, 0.40)
fig, axs = plt.subplots(1, len(show), figsize=(14, 3.8), sharex=True, sharey=True)
for ax, e in zip(axs, show):
    for ev, col in (("T", "#e53935"), ("M", "#1e88e5")):
        s = [r for r in GCL if r["event"] == ev]
        ax.plot([r["K"] for r in s], [r[f"K_closed_{e:g}"] for r in s], ".", color=col, ms=5)
    lim = (-35, 15); ax.plot(lim, lim, "k:"); ax.set(xlim=lim, ylim=lim, title=f"K_closed, ε = {e:g}", xlabel="K")
axs[0].set_ylabel("K_closed")

In [ ]:
# (c) what moves the deficit? f·(h*500 − h̄_e) = below + above, split at the LCL
E = 0.20
rows = {}
for k in named:
    r = CRES[k]
    ff = r[f"f_{E:g}"]
    rows[k] = dict(f=ff, L=r[f"L_{E:g}"], factor=ff / (1 - ff), K_hstar=r["K_hstar"], K_hbar=r[f"K_hbar_{E:g}"],
                   K_below_LCL=r[f"K_below_{E:g}"], K_above_LCL=r[f"K_above_{E:g}"], K_closed=r[f"K_closed_{E:g}"], K=r["K"])
print(f"ε = {E:g}. K < 0 means the value rises with RH. below + above = K of f·(h*500 − h̄_e); each includes the change in weight as the LCL moves.")
pd.DataFrame(rows).T

## Paste back
Sections 1, 3, 4, 7 and 10 are the ones to read together first.